In [42]:
# Cell 1 — Đường dẫn và thư viện
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(r"D:\OLAP")
DATA_DIR = PROJECT_ROOT / "Data" / "Processed"

COLLISIONS_FILE = DATA_DIR / "collisions_clean.csv"
VEHICLES_FILE = DATA_DIR / "vehicles_clean.csv"
CASUALTIES_FILE = DATA_DIR / "casualties_clean.csv"

In [43]:
# Cell 2 — Đọc ba file Clean
COLLISION_COLUMNS = [
    "collision_id", "collision_year", "collision_date", "collision_time",
    "local_authority_district_code", "collision_severity",
    "first_road_class", "road_type", "speed_limit_mph", "area_type",
    "junction_detail", "junction_control", "pedestrian_crossing",
    "light_conditions", "weather_conditions", "road_surface_conditions",
]

VEHICLE_COLUMNS = [
    "collision_id", "vehicle_reference", "vehicle_type",
    "driver_age_band", "driver_sex", "vehicle_manoeuvre",
    "skidding_and_overturning", "vehicle_leaving_carriageway",
]

CASUALTY_COLUMNS = [
    "collision_id", "vehicle_reference", "casualty_reference",
    "casualty_class", "casualty_sex", "casualty_age_band",
    "casualty_severity", "casualty_type",
    "pedestrian_location", "pedestrian_movement",
]

def read_clean(path, columns):
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy: {path}")

    return pd.read_csv(
        path,
        usecols=columns,
        dtype="string",
        encoding="utf-8-sig",
        keep_default_na=False,
        na_filter=False,
        low_memory=False,
    )

df_col = read_clean(COLLISIONS_FILE, COLLISION_COLUMNS)
df_veh = read_clean(VEHICLES_FILE, VEHICLE_COLUMNS)
df_cas = read_clean(CASUALTIES_FILE, CASUALTY_COLUMNS)

print(f"Collisions: {len(df_col):,}")
print(f"Vehicles  : {len(df_veh):,}")
print(f"Casualties: {len(df_cas):,}")

Collisions: 513,801
Vehicles  : 937,265
Casualties: 652,821


In [44]:
# Cell 3 — Lọc vehicle mà FactCasualty thật sự tham chiếu
VEHICLE_KEY = ["collision_id", "vehicle_reference"]

if df_veh.duplicated(VEHICLE_KEY).any():
    raise ValueError("Vehicles có natural key bị trùng.")

fact_vehicle_keys = df_cas[VEHICLE_KEY].drop_duplicates()

df_fact_vehicle = df_veh.merge(
    fact_vehicle_keys,
    on=VEHICLE_KEY,
    how="inner",
    validate="one_to_one",
)

print(f"Vehicle nguồn                  : {len(df_veh):,}")
print(f"Vehicle được Fact tham chiếu   : {len(df_fact_vehicle):,}")
print(f"Vehicle ngoài phạm vi Fact     : {len(df_veh) - len(df_fact_vehicle):,}")

Vehicle nguồn                  : 937,265
Vehicle được Fact tham chiếu   : 566,945
Vehicle ngoài phạm vi Fact     : 370,320


In [45]:
# Cell 4 — Quy tắc Unknown và hàm đếm Dimension
UNKNOWN_VALUES = {
    "", "-1", "unknown", "not known", "not reported",
    "data missing or out of range", "unallocated",
    "undefined", "nan", "<na>", "null",
}

def canonicalize(series):
    value = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    normalized = value.str.casefold()
    missing = value.isna() | normalized.isin(UNKNOWN_VALUES)

    return value.mask(missing, "Unknown")

def profile_dimension(name, dataframe, grain_columns, not_applicable=0):
    members = dataframe[grain_columns].copy()

    for column in grain_columns:
        members[column] = canonicalize(members[column])

    members = members.drop_duplicates()

    unknown_flags = members.apply(
        lambda column: column.eq("Unknown")
    )

    all_unknown = unknown_flags.all(axis=1)
    any_unknown = unknown_flags.any(axis=1)

    all_unknown_count = int(all_unknown.sum())
    partial_unknown_count = int((any_unknown & ~all_unknown).sum())
    business_members = len(members) - all_unknown_count
    expected_rows = business_members + 1 + not_applicable

    return {
        "Dimension": name,
        "Distinct combinations": len(members),
        "Partial Unknown": partial_unknown_count,
        "All Unknown": all_unknown_count,
        "Business members": business_members,
        "Unknown member": 1,
        "Not Applicable": not_applicable,
        "Expected rows": expected_rows,
    }

results = []

def add_dimension(name, dataframe, columns, not_applicable=0):
    results.append(
        profile_dimension(name, dataframe, columns, not_applicable)
    )

In [46]:
# Cell 5 — DimDate, DimTimeBand, DimTime, DimRegion
# DimDate: toàn bộ ngày lịch trong phạm vi 2021–2025, cộng Unknown.
date_count = len(pd.date_range("2021-01-01", "2025-12-31", freq="D"))
results.append({
    "Dimension": "DimDate",
    "Distinct combinations": date_count,
    "Partial Unknown": 0,
    "All Unknown": 0,
    "Business members": date_count,
    "Unknown member": 1,
    "Not Applicable": 0,
    "Expected rows": date_count + 1,
})

# Bảy khung giờ nghiệp vụ; TimeBandSort là thứ tự hiển thị.
TIME_BANDS = pd.DataFrame(
    [
        ("Overnight", 0, 4, 1),
        ("Early Morning", 5, 6, 2),
        ("Morning Rush", 7, 9, 3),
        ("Working Hours", 10, 15, 4),
        ("Evening Rush", 16, 18, 5),
        ("Evening", 19, 22, 6),
        ("Late Night", 23, 23, 7),
    ],
    columns=["TimeBand", "StartHour", "EndHour", "TimeBandSort"],
)

results.append({
    "Dimension": "DimTimeBand",
    "Distinct combinations": len(TIME_BANDS),
    "Partial Unknown": 0,
    "All Unknown": 0,
    "Business members": len(TIME_BANDS),
    "Unknown member": 1,
    "Not Applicable": 0,
    "Expected rows": len(TIME_BANDS) + 1,
})

# DimTime: mỗi phút trong ngày, cộng Unknown.
time_count = 24 * 60
results.append({
    "Dimension": "DimTime",
    "Distinct combinations": time_count,
    "Partial Unknown": 0,
    "All Unknown": 0,
    "Business members": time_count,
    "Unknown member": 1,
    "Not Applicable": 0,
    "Expected rows": time_count + 1,
})

# 9 vùng của England, Wales và Scotland; cộng Unknown.
REGIONS = [
    "North East England", "North West England",
    "Yorkshire and The Humber", "East Midlands",
    "West Midlands", "East of England", "London",
    "South East England", "South West England",
    "Wales", "Scotland",
]

results.append({
    "Dimension": "DimRegion",
    "Distinct combinations": len(REGIONS),
    "Partial Unknown": 0,
    "All Unknown": 0,
    "Business members": len(REGIONS),
    "Unknown member": 1,
    "Not Applicable": 0,
    "Expected rows": len(REGIONS) + 1,
})

In [47]:
# Cell 6 — DimLocalAuthority, DimRoad, DimEnvironment, DimJunction
add_dimension(
    "DimLocalAuthority",
    df_col,
    ["local_authority_district_code"],
)

add_dimension(
    "DimRoad",
    df_col,
    ["first_road_class", "road_type", "speed_limit_mph", "area_type"],
)

add_dimension(
    "DimEnvironment",
    df_col,
    ["light_conditions", "weather_conditions", "road_surface_conditions"],
)

add_dimension(
    "DimJunction",
    df_col,
    ["junction_detail", "junction_control", "pedestrian_crossing"],
)

In [48]:
# Cell 7 — DimSeverity, DimPedestrianContext, DimCasualtyProfile, DimVehicleProfile
# Một DimSeverity dùng cho hai vai trò: collision severity và casualty severity.
severity_values = pd.concat(
    [df_col["collision_severity"], df_cas["casualty_severity"]],
    ignore_index=True,
)
add_dimension(
    "DimSeverity",
    pd.DataFrame({"severity": severity_values}),
    ["severity"],
)

# Chỉ profile context từ casualty được phân loại là Pedestrian.
pedestrian = df_cas.loc[
    df_cas["casualty_class"].str.casefold().eq("pedestrian"),
    ["pedestrian_location", "pedestrian_movement"],
].copy()

# “Not a Pedestrian” không phù hợp với nhóm đã xác định là Pedestrian.
for column in ["pedestrian_location", "pedestrian_movement"]:
    pedestrian[column] = canonicalize(pedestrian[column])
    pedestrian.loc[
        pedestrian[column].str.casefold().eq("not a pedestrian"),
        column,
    ] = "Unknown"

add_dimension(
    "DimPedestrianContext",
    pedestrian,
    ["pedestrian_location", "pedestrian_movement"],
    not_applicable=1,
)

add_dimension(
    "DimCasualtyProfile",
    df_cas,
    ["casualty_age_band", "casualty_sex", "casualty_class", "casualty_type"],
)

add_dimension(
    "DimVehicleProfile",
    df_fact_vehicle,
    [
        "vehicle_type",
        "driver_age_band",
        "driver_sex",
        "vehicle_manoeuvre",
        "skidding_and_overturning",
        "vehicle_leaving_carriageway",
    ],
)

In [49]:
# Cell 8 — Bảng tổng hợp số dòng dự kiến
dimension_order = [
    "DimDate",
    "DimTimeBand",
    "DimTime",
    "DimRegion",
    "DimLocalAuthority",
    "DimRoad",
    "DimEnvironment",
    "DimJunction",
    "DimSeverity",
    "DimPedestrianContext",
    "DimCasualtyProfile",
    "DimVehicleProfile",
]

summary = pd.DataFrame(results)
summary["Dimension"] = pd.Categorical(
    summary["Dimension"],
    categories=dimension_order,
    ordered=True,
)
summary = summary.sort_values("Dimension").reset_index(drop=True)

if len(summary) != 12:
    raise ValueError(f"Cần 12 Dimension, hiện tính được {len(summary)}.")

display(summary)

,Dimension,Distinct combinations,Partial Unknown,All Unknown,Business members,Unknown member,Not Applicable,Expected rows
0,DimDate,1826,0,0,1826,1,0,1827
1,DimTimeBand,7,0,0,7,1,0,8
2,DimTime,1440,0,0,1440,1,0,1441
3,DimRegion,11,0,0,11,1,0,12
4,DimLocalAuthority,370,0,0,370,1,0,371
5,DimRoad,385,81,0,385,1,0,386
6,DimEnvironment,228,87,1,227,1,0,228
7,DimJunction,295,115,1,294,1,0,295
8,DimSeverity,3,0,0,3,1,0,4
9,DimPedestrianContext,84,17,1,83,1,1,85
